# TP 3

In [86]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, accuracy_score

# Exercice 1 – Matrices de confusion et risque empirique

## 1) On calcule les matrices de confusions des deux modèles :

- les lignes correspondent aux classes réelles
- les colonnes correspondent aux classes prédites.

Modèle $g_1$:

$$C_1 = \begin{pmatrix} 3 & 2 \\ 1 & 4 \end{pmatrix}$$

Modèle $g_2$ :

$$C_2 = \begin{pmatrix} 4 & 1 \\ 3 & 2 \end{pmatrix}$$

## 2) On calcule le risque empirique pour la perte $L_1$ :


- $g_1$ : $3 \times 2 + 1 \times 1 = 7$, donc $\hat R_{L_1}(g_1) = 7/10$
- $g_2$ : $3 \times 1 + 1 \times 3 = 6$, donc $\hat R_{L_1}(g_2) = 6/10$

C'est le modèle $g_2$ qui minimise le risque empirique.

## 3) Risque empirique pour la perte 0-1

- $g_1$ : $2 + 1 = 3$ erreurs, donc $\hat R_{0\text{-}1}(g_1) = 3/10$
- $g_2$ : $1 + 3 = 4$ erreurs, donc $\hat R_{0\text{-}1}(g_2) = 4/10$

Pour la perte 0-1, le meilleur modèle est $g_1$.

## 4) Comaraison des deux conclusions

- avec la perte 0-1, $g_1$ est meilleur
- avec la perte $L_1$, $g_2$ est meilleur

La perte 0-1 compte toutes les erreurs de la même façon.
La perte $L_1$ punit un faux positif 3 fois plus qu'un faux négatif. $g_1$ fait 2 faux positifs alors que $g_2$ n'en fait qu'un seul. $g_2$ fait plus d'erreurs, mais ce sont des erreurs moins "importantes".

Le choix de la fonction de perte dépend donc de ce qu'on veut le plus pénaliser comme erreur.

## Question 5 – Vérification en Python

In [87]:
g1 = np.array([ 1,  1, -1,  1,  1,  1, -1, -1, -1,  1])
g2 = np.array([ 1, -1,  1, -1, -1, -1, -1, -1,  1, -1])
y  = np.array([ 1,  1,  1,  1,  1, -1, -1, -1, -1, -1])
N = 10

In [88]:
C1 = confusion_matrix(y, g1, labels=[-1, 1])
C2 = confusion_matrix(y, g2, labels=[-1, 1])

print(C1)
print(C2)

[[3 2]
 [1 4]]
[[4 1]
 [3 2]]


In [89]:
risque_L1_g1 = (3 * C1[0, 1] + 1 * C1[1, 0]) / N
risque_L1_g2 = (3 * C2[0, 1] + 1 * C2[1, 0]) / N

print("Risque L1 de g1 :", risque_L1_g1)
print("Risque L1 de g2 :", risque_L1_g2)

Risque L1 de g1 : 0.7
Risque L1 de g2 : 0.6


In [90]:
risque_01_g1 = 1 - accuracy_score(y, g1)
risque_01_g2 = 1 - accuracy_score(y, g2)

print("Risque 0-1 de g1 :", risque_01_g1)
print("Risque 0-1 de g2 :", risque_01_g2)

Risque 0-1 de g1 : 0.30000000000000004
Risque 0-1 de g2 : 0.4


# Exercice 2 – Comparaison automatique de deux modèles

## 1) Risque empirique 0-1 à partir de $C$

On écrit une fonction pour clculer le risque empirique associé à la fonction de perte 0-1 à partir de la matrice de confusion C

In [91]:
def risque_01(C):
    erreurs = C[0, 1] + C[1, 0]
    total = C.sum()
    return erreurs / total

## 2) On calcule le risque des deux modèles et on compare

In [92]:
def meilleur_modele(C1, C2):
    r1 = risque_01(C1)
    r2 = risque_01(C2)
    if r1 < r2:
        return "g1 a le plus faible risque"
    elif r2 < r1:
        return "g2 a le plus faible risque"
    else:
        return "g1 et g2 ont le même risque"

## 3) On teste sur les matrices de l'exercice 1

In [93]:
print("Risque 0-1 de g1 :", risque_01(C1))
print("Risque 0-1 de g2 :", risque_01(C2))
print(meilleur_modele(C1, C2))

Risque 0-1 de g1 : 0.3
Risque 0-1 de g2 : 0.4
g1 a le plus faible risque


## 4) On adapte pour une perte quelconque

In [94]:
def risque_cout(C, cout):
    total_cout = 0
    for i in range(2):
        for j in range(2):
            total_cout = total_cout + C[i, j] * cout[i, j]
    return total_cout / C.sum()

On test sur L1

In [95]:
cout_L1 = np.array([[0, 3],
                    [1, 0]])

print("Risque L1 de g1 :", risque_cout(C1, cout_L1))
print("Risque L1 de g2 :", risque_cout(C2, cout_L1))

Risque L1 de g1 : 0.7
Risque L1 de g2 : 0.6


## 5) Le coût de l'observation ne dépend que de la case de la matrice de confusion dans laquelle elle tombe. Deux observations qui sont dans la même case ont le même coût.

### Le risque empirique est donc : (nombre d'observations dans chaque case × coût de cette case), additionné sur les cases, puis divisé par $N$.

# Exercice 3 – Comparaison sur apprentissage et test

## 1) On génère le vecteur et les deux autres en de prédiction en changeant de manière aléatoire 10% et 20% des étqiuettes

In [96]:
def inverser(y, taux):
    pred = y.copy()
    n = int(taux * len(y))
    positions = np.random.choice(len(y), n, replace=False)
    pred[positions] = -pred[positions]
    return pred

In [97]:
np.random.seed(0)

N = 100
y_train = np.random.choice([-1, 1], size=N)

pred1_train = inverser(y_train, 0.10)
pred2_train = inverser(y_train, 0.20)

## 2) On calcule les matrices de confusion et risque 0-1 sur l'apprentissage

In [98]:
C1_train = confusion_matrix(y_train, pred1_train, labels=[-1, 1])
C2_train = confusion_matrix(y_train, pred2_train, labels=[-1, 1])

print("Matrice de confusion de g1 :")
print(C1_train)
print("Matrice de confusion de g2 :")
print(C2_train)

r01_g1_train = risque_01(C1_train)
r01_g2_train = risque_01(C2_train)

print("Risque 0-1 de g1 :", r01_g1_train)
print("Risque 0-1 de g2 :", r01_g2_train)

Matrice de confusion de g1 :
[[42  2]
 [ 8 48]]
Matrice de confusion de g2 :
[[34 10]
 [10 46]]
Risque 0-1 de g1 : 0.1
Risque 0-1 de g2 : 0.2


Le meilleur modèle sur l'apprentissage est donc $g_1$.

## 3) On fait le test, $g_1$ se trompe sur 25 % des étiquettes et $g_2$ sur 15 %.

In [99]:
N_test = 100
y_test = np.random.choice([-1, 1], size=N_test)

pred1_test = inverser(y_test, 0.25)
pred2_test = inverser(y_test, 0.15)

C1_test = confusion_matrix(y_test, pred1_test, labels=[-1, 1])
C2_test = confusion_matrix(y_test, pred2_test, labels=[-1, 1])

print("Matrice de confusion de g1 :")
print(C1_test)
print("Matrice de confusion de g2 :")
print(C2_test)

r01_g1_test = risque_01(C1_test)
r01_g2_test = risque_01(C2_test)

print("Risque 0-1 de g1 :", r01_g1_test)
print("Risque 0-1 de g2 :", r01_g2_test)

Matrice de confusion de g1 :
[[35 13]
 [12 40]]
Matrice de confusion de g2 :
[[40  8]
 [ 7 45]]
Risque 0-1 de g1 : 0.25
Risque 0-1 de g2 : 0.15


Sur l'ensemble de test, $g_1$ fait 25 erreurs et $g_2$ fait 15 erreurs. Le meilleur modèle sur le test est donc $g_2$.

Sur l'apprentissage $g_1$ est meilleur que $g_2$, mais sur le test c'est l'inverse.

## 4) On utilise maintenant la perte asymétrique $L_1$

In [100]:
def risque_L1(y, pred):
    total = 0
    for i in range(len(y)):
        if pred[i] == -1 and y[i] == 1:
            total = total + 1
        elif pred[i] == 1 and y[i] == -1:
            total = total + 3
    return total / len(y)

In [101]:
L1_g1_train = risque_L1(y_train, pred1_train)
L1_g2_train = risque_L1(y_train, pred2_train)
L1_g1_test = risque_L1(y_test, pred1_test)
L1_g2_test = risque_L1(y_test, pred2_test)

print("Risque L1 de g1 (apprentissage) :", L1_g1_train)
print("Risque L1 de g2 (apprentissage) :", L1_g2_train)
print("Risque L1 de g1 (test) :", L1_g1_test)
print("Risque L1 de g2 (test) :", L1_g2_test)

Risque L1 de g1 (apprentissage) : 0.14
Risque L1 de g2 (apprentissage) : 0.4
Risque L1 de g1 (test) : 0.51
Risque L1 de g2 (test) : 0.31


## 5) On fait les tableaux comparatifs

In [102]:
tableau_01 = pd.DataFrame({"g1": [r01_g1_train, r01_g1_test],
                           "g2": [r01_g2_train, r01_g2_test]},
                          index=["Apprentissage", "Test"])

tableau_L1 = pd.DataFrame({"g1": [L1_g1_train, L1_g1_test],
                           "g2": [L1_g2_train, L1_g2_test]},
                          index=["Apprentissage", "Test"])

print("Perte 0-1")
print(tableau_01)
print()
print("Perte L1")
print(tableau_L1)

Perte 0-1
                 g1    g2
Apprentissage  0.10  0.20
Test           0.25  0.15

Perte L1
                 g1    g2
Apprentissage  0.14  0.40
Test           0.51  0.31


$g_1$ est le meilleur sur l'apprentissage et $g_2$ est le meilleur sur le test, pour les deux fonctions de perte. Un modèle très bon sur les données d'apprentissage n'est pas forcément bon sur de nouvelles données. Donc non le modèle ayant le plus faible risque sur l’ensemble d’apprentissage n'est pas nécessairement celui qui présente le plus faible risque sur l’ensemble de test.

## 6) On refait toute l'expérience pour plusieurs graines.

In [103]:
for graine in range(10):
    np.random.seed(graine)

    y_train = np.random.choice([-1, 1], size=100)
    pred1_train = inverser(y_train, 0.10)
    pred2_train = inverser(y_train, 0.20)

    y_test = np.random.choice([-1, 1], size=100)
    pred1_test = inverser(y_test, 0.25)
    pred2_test = inverser(y_test, 0.15)

    a1 = np.mean(pred1_train != y_train)
    a2 = np.mean(pred2_train != y_train)
    b1 = np.mean(pred1_test != y_test)
    b2 = np.mean(pred2_test != y_test)

    c1 = risque_L1(y_train, pred1_train)
    c2 = risque_L1(y_train, pred2_train)
    d1 = risque_L1(y_test, pred1_test)
    d2 = risque_L1(y_test, pred2_test)

    print("  0-1 // apprentissage : g1 =", a1, ", g2 =", a2, "| test : g1 =", b1, ", g2 =", b2)
    print("  L1  // apprentissage : g1 =", c1, ", g2 =", c2, "| test : g1 =", d1, ", g2 =", d2)

  0-1 // apprentissage : g1 = 0.1 , g2 = 0.2 | test : g1 = 0.25 , g2 = 0.15
  L1  // apprentissage : g1 = 0.14 , g2 = 0.4 | test : g1 = 0.51 , g2 = 0.31
  0-1 // apprentissage : g1 = 0.1 , g2 = 0.2 | test : g1 = 0.25 , g2 = 0.15
  L1  // apprentissage : g1 = 0.14 , g2 = 0.36 | test : g1 = 0.55 , g2 = 0.27
  0-1 // apprentissage : g1 = 0.1 , g2 = 0.2 | test : g1 = 0.25 , g2 = 0.15
  L1  // apprentissage : g1 = 0.22 , g2 = 0.48 | test : g1 = 0.49 , g2 = 0.33
  0-1 // apprentissage : g1 = 0.1 , g2 = 0.2 | test : g1 = 0.25 , g2 = 0.15
  L1  // apprentissage : g1 = 0.24 , g2 = 0.4 | test : g1 = 0.51 , g2 = 0.27
  0-1 // apprentissage : g1 = 0.1 , g2 = 0.2 | test : g1 = 0.25 , g2 = 0.15
  L1  // apprentissage : g1 = 0.2 , g2 = 0.42 | test : g1 = 0.51 , g2 = 0.27
  0-1 // apprentissage : g1 = 0.1 , g2 = 0.2 | test : g1 = 0.25 , g2 = 0.15
  L1  // apprentissage : g1 = 0.2 , g2 = 0.42 | test : g1 = 0.53 , g2 = 0.31
  0-1 // apprentissage : g1 = 0.1 , g2 = 0.2 | test : g1 = 0.25 , g2 = 0.15
  L1

Pour 0-1, les risques sont toujours les mêmes.

Pour la perte $L_1$, les risques changent, dépend des étiquettes tirées.

Cette expérience permet d'illustrer que le risque empirique dépend des données utilisées. Le risque sur l'ensemble d'apprentissage ne suffit pas pour choisir un modèle : le meilleur sur l'apprentissage peut être moins bon sur de nouvelles données.